# Comparing paired data 
## Introduction


## Preparing data for hypothesis testing
### Importing the paired data


In [ ]:
import pandas as pd

# Each row represents a paired observation; unit is inches
crossed_fertil = pd.Series(
    [23.5, 12, 21, 22, 19.125, 21.5, 22.125, 20.375, 18.250, 
     21.625, 23.250, 21, 22.125, 23, 12],
    name='crossed_fertil')
self_fertil = pd.Series(
    [17.375, 20.375, 20, 20, 18.375, 18.625, 18.625, 15.25, 
     16.5, 18, 16.25, 18, 12.75, 15.5, 18],
    name='self_fertil')

# Combine the data in a DataFrame
data_fertil = pd.merge(
    crossed_fertil, self_fertil, left_index=True, right_index=True)

print("First rows of the fertilization DataFrame:")
print(data_fertil.head())

### Descriptive statistics


In [ ]:
# Calculate the difference between cross-fertilized and self-fertilized
# for each pair using broadcasting under the hood
# differences_fertil = (
#     data_fertil['crossed_fertil'] - data_fertil['self_fertil'])
differences_fertil = crossed_fertil - self_fertil  # Using Series

# Print or display the calculated differences
print("Paired growth differences between\n cross- and self-fertilization:")
print(differences_fertil)

In [ ]:
import scipy.stats as stats

# Descriptive statistics of the differences
differences_fertil_stats = stats.describe(differences_fertil)

print("Descriptive statistics for the differences:")
print(pd.Series(differences_fertil_stats._asdict()))

### Data visualization


In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import pingouin as pg

pd.set_option('future.no_silent_downcasting', True)

_, axes = plt.subplots(nrows=1, ncols=2, figsize=(6, 3), width_ratios=(1,2))

# Create the stripplot

sns.stripplot(
    y=differences_fertil,
    ax=axes[0])

# Add a boxplot on the right side
sns.boxplot(
    y=differences_fertil,
    ax=axes[0],
    width=0.25,
    showmeans=True,                  # Show mean inside the boxplot
    fliersize=0,                     # Hide outliers from boxplot
    boxprops={'facecolor': 'none'},  # Make the boxplot transparent
    medianprops={'color': 'black'},
    meanprops={
        'marker': 'o',
        'markerfacecolor': 'red',
        'markeredgecolor': 'black'},)

# Add a horizontal line at y=0 (no difference)
axes[0].axhline(
    y=0,
    color='gray', linestyle='--', linewidth=1.5)

# Set plot labels and title
axes[0].set_ylabel('Difference in height (in)')
axes[0].set_xticks([])
axes[0].set_title('Boxplot')

# Create the stripplot

pg.plot_paired(
    # We need to reshape the DataFrame in long format
    data=data_fertil.reset_index().melt(
        value_vars=['crossed_fertil', 'self_fertil'],
        id_vars='index',
        var_name='fertilization',
        value_name='plant_height'),
    dv='plant_height',
    within='fertilization',
    subject='index',
    ax=axes[1],
    boxplot=True,
    orient='v',
    # boxplot_in_front=False,  # fix submitted via a PR
    boxplot_kwargs={'color': 'white', 'linewidth': 2, 'zorder': 1},)

axes[1].set_xticks([0, 1], ['crossed-fertilized', 'self-fertilized'])
axes[1].set_xlabel(None)
axes[1].set_ylabel("Plant height (in)")
axes[1].set_title("Paired plot")
sns.despine(ax=axes[1], trim=True)  # not default in Pingouin anymore

plt.tight_layout(); 

### Assessing assumptions
#### Normality test


In [ ]:
import warnings
warnings.filterwarnings('ignore')

print("Normality tests for the differences\n between paired observations:")
print('-'*40)
print("D'Agostino-Pearson:")
print(pg.normality(differences_fertil, method='normaltest'))
print()  # Print blank separation line
print("Shapiro-Wilk:")
print(pg.normality(differences_fertil, method='shapiro'))


#### Q-Q plot


In [ ]:

plt.figure(figsize=(3.5, 3))
pg.qqplot(differences_fertil)
plt.title("Paired differences");

#### Discusssion on normality testing
#### Note on homoscedasticity


## Assessing the significance of paired difference mean with t-test


### The t-ratio


#### Manual calculation


In [ ]:
import numpy as np

# Calculate the mean of the paired differences
differences_fertil_mean = differences_fertil.mean()

# Calculate the standard error of the mean difference
differences_fertil_stats_se = np.sqrt(
    differences_fertil_stats.variance / differences_fertil.size)

# Calculate the t-statistic for the paired t-test
t_statistic_paired = differences_fertil_mean / differences_fertil_stats_se

# Calculate the degrees of freedom
df_paired = differences_fertil.size - 1

# Print the results
print(f"Manually calculated t-statistic (paired) = \
{t_statistic_paired:.4f} with {df_paired} degrees of freedom")

#### P value


In [ ]:
# Calculate the P value using the t-distribution (two-sided test)
p_value_paired = 2 * (1 - stats.t.cdf(abs(t_statistic_paired), df_paired))

# Print the results
print(f"P value for the paired t-test = {p_value_paired:.4f}")

#### Visualizing the test results


In [ ]:

# Significance level (alpha)
α = 0.05

# Calculate critical t-values (two-tailed test)
t_crit_lower = stats.t.ppf(α/2, df_paired)
t_crit_upper = stats.t.ppf(1 - α/2, df_paired)

# Generate x values for plotting
x = np.linspace(-5, 5, 1000)
hx = stats.t.pdf(x, df_paired)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, hx, lw=2, color="black")

# Plot the critical t-values
plt.axvline(
    x=t_crit_lower,
    color='orangered', linestyle='--')

plt.axvline(
    x=t_crit_upper,
    color='orangered', linestyle='--',
    label=f't* ({t_crit_upper:.3f})')

# Shade the rejection regions (alpha)
plt.fill_between(
    x[x <= t_crit_lower],
    hx[x <= t_crit_lower],
    linestyle="-", linewidth=2, color='tomato',
    label=f'α ({α})')

plt.fill_between(
    x[x >= t_crit_upper],
    hx[x >= t_crit_upper],
    linestyle="-", linewidth=2,
    color='tomato')

# Plot the observed t-statistic
plt.axvline(
    x=t_statistic_paired,
    color='limegreen', linestyle='-.',
    label=f't ({t_statistic_paired:.3f})')

# Shade the P-value areas (two-tailed)
plt.fill_between(
    x[x <= -abs(t_statistic_paired)],
    hx[x <= -abs(t_statistic_paired)],
    color='greenyellow',
    label=f'P ({p_value_paired:.4f})')

plt.fill_between(
    x[x >= abs(t_statistic_paired)],
    hx[x >= abs(t_statistic_paired)],
    color='greenyellow')

# Add labels and title
plt.xlabel('t')
plt.ylabel('Density')
plt.margins(x=0, y=0)
plt.yticks([])
plt.title(f"t-distribution (DF={df_paired})")
plt.legend();

#### Confidence interval


In [ ]:
# Print floating point numbers instead of scientific notation
np.set_printoptions(suppress=True)

# Calculate the confidence interval (e.g., 1 - α = 95% confidence)
margin_of_error_paired = abs(
    stats.t.ppf((1 + (1 - α))/2, df_paired)) * differences_fertil_stats_se

ci_paired = np.array([
    differences_fertil_mean - margin_of_error_paired,
    differences_fertil_mean + margin_of_error_paired])

# Print the results
print(
    "95% CI for the mean difference (crossed - self fertilization):",
    ci_paired.round(3))

### Performing paired t-test in Python


In [ ]:
# Paired t-test using SciPy
t_statistic_scipy_paired, p_value_scipy_paired = stats.ttest_rel(
    crossed_fertil, self_fertil)

# Paired t-test using Pingouin
ttest_results_pingouin_paired = pg.ttest(
    x=crossed_fertil, y=self_fertil, paired=True)

# Print the results
print("Paired t-test results (SciPy):")
print(f"t-statistic = {t_statistic_scipy_paired:.3f}, \
P value = {p_value_scipy_paired:.4f}")
print()
print("Paired t-test results (Pingouin):")
print(ttest_results_pingouin_paired.round(4))

### Follow-up analyses
#### Effect size


In [ ]:
# Calculate Cohen's d manually
cohens_d_manual_paired = (
    differences_fertil_mean/differences_fertil_stats.variance**.5)
# cohens_d_manual_paired = (
#     differences_fertil.mean() / differences_fertil.std(ddof=1)

# Calculate unbiased Cohen's d using pingouin for paired t-test
effect_size_pingouin_paired = pg.compute_effsize(
    crossed_fertil, self_fertil, paired=True, eftype='cohen')

print(f"Cohen's d (manual, paired) = {cohens_d_manual_paired:.3f}")
print(f"Unbiased Cohen's d (Pingouin, paired) = \
{effect_size_pingouin_paired:.3f}")

#### Evaluating the effectiveness of pairing


In [ ]:

# Calculate the Pearson correlation coefficient
pearson_r = np.corrcoef(crossed_fertil, self_fertil)[0, 1]

# Create the scatterplot with regression line
plt.figure(figsize=(3.5, 3))
ax = sns.regplot(
    x=crossed_fertil,
    y=self_fertil,
    line_kws={'label':f'Pearson r = {pearson_r:.3f}'})

# Set plot labels and title
plt.title('Effectiveness of pairing')
plt.xlabel('Cross-fertilized plant height (in)')
plt.ylabel('Self-fertilized plant height (in)')
plt.legend(frameon=False);

### Extensions of paired t-test
#### One-sided t-test


In [ ]:
print("One-sided paired t-test (Pingouin):")
print(
    pg.ttest(
        crossed_fertil, self_fertil,
        paired=True,
        alternative='greater')
    .round(3))

#### Ratio paired t-test


In [ ]:
# Data
control_cells = pd.Series([24,  6, 16, 5, 2], name='control')
treated_cells = pd.Series([52, 11, 28, 8, 4], name='treated')

# Create a DataFrame for Pingouin's plot_paired function
data_cells = pd.merge(
    control_cells, treated_cells, left_index=True, right_index=True)

print("Enzyme activity of cells:")
print(data_cells)

In [ ]:

# Need a long/melted format of the dataframe for pg.plot_paired
data_cells_long = data_cells.reset_index().melt(
        value_vars=['control', 'treated'],
        id_vars='index',
        var_name='condition',
        value_name='enzyme_activity')

# Create the figure with two subplots
_, axes = plt.subplots(nrows=1, ncols=2, figsize=(6, 3))

# Left subplot: before-after plot with original data
pg.plot_paired(
    data=data_cells_long,
    dv="enzyme_activity",
    within="condition",
    subject="index",
    boxplot_kwargs={'color': 'white', 'linewidth': 2, 'zorder': 1},
    ax=axes[0])
axes[0].set_title('original data')
axes[0].set_ylabel('Enzyme activity')
axes[0].set_xlabel('')

# Right subplot: before-after plot with log-transformed data
# Apply log transformation to the 'enzyme_activity' column
data_cells_long['log_enzyme_activity'] = np.log(
    data_cells_long['enzyme_activity'])

pg.plot_paired(
    data=data_cells_long,
    dv="log_enzyme_activity",
    within="condition",
    subject="index",
    boxplot_kwargs={'color': 'white', 'linewidth': 2, 'zorder': 1},
    ax=axes[1])
axes[1].set_title('log-transformed data')
axes[1].set_ylabel(r'Enzyme activity ($\log$)')
axes[1].set_xlabel('')

sns.despine(trim=True)
plt.tight_layout();


In [ ]:
# Perform the paired t-test on the log-transformed data
# Remember we are analyzing \bar{x} - \bar{y}
ratio_ttest_results_paired = pg.ttest(
    x=treated_cells.apply(np.log10),
    y=control_cells.apply(np.log10),
    paired=True)

print("Ratio paired t-test on enzyme activity (Pingouin):")
print(ratio_ttest_results_paired.round(3))

In [ ]:
# Extract the mean difference and confidence interval from the Pingouin results
ratio_mean_diff_log = (
    treated_cells.apply(np.log10) - control_cells.apply(np.log10)).mean()
ratio_ci_log = ratio_ttest_results_paired.iloc[0,4]

# Calculate the mean ratio and its confidence interval on the original scale
ratio_factor = 10 ** ratio_mean_diff_log
ratio_ci_original = np.power(10, ratio_ci_log)  # Same as 10 ** ratio_ci_log

# Print the results
print(
    f"On average, the treatment multiplies activity by {ratio_factor:.2f} \
translating to\n{100*(ratio_factor - 1):.1f}% increase, with 95% CI on the \
original scale of", ratio_ci_original.round(3))

#### McNemar test
##### How the McNemar test works


##### Case-control study
##### McNemar test step-by-step


In [ ]:
table_mcnemar = np.array([
    [13, 4],
    [25, 92]])

# Extract discordant pair counts from the contingency table
b_mcnemar = table_mcnemar[0, 1]  # Count cases exposed, controls not exposed
c_mcnemar = table_mcnemar[1, 0]  # Count cases not exposed, controls exposed

# Calculate the McNemar test statistic with Yates continuity correction
mcnemar_statistic = (
    abs(b_mcnemar - c_mcnemar) - 1)**2 / (b_mcnemar + c_mcnemar)

# Print the calculated statistic
print("Table of paired data:")
print(table_mcnemar)
print()
print("McNemar's M²:", round(mcnemar_statistic, 3))

In [ ]:

from scipy.stats import chi2

# Significance level (alpha)
α = 0.05

# Calculate critical chi-squared value
chi2_mcnemar = chi2(df=1)  # 1 degree of freedom
chi2_crit = chi2_mcnemar.ppf(1 - α)

# Calculate the P value
p_value_mcnemar = 1 - chi2_mcnemar.cdf(x=mcnemar_statistic)

# Generate x values for plotting
x = np.linspace(0, 10, 1000)
hx = chi2_mcnemar.pdf(x)  # χ² PDF with 1 degree of freedom

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, hx, lw=2, color="black")

# Plot the critical chi-squared value
plt.axvline(
    x=chi2_crit,
    color='orangered', linestyle='--',
    label=f"χ²* ({chi2_crit:.3f})")

# Shade the rejection region (alpha)
plt.fill_between(
    x[x >= chi2_crit],
    hx[x >= chi2_crit],
    linestyle="-", linewidth=2, color='tomato',
    label=f'α ({α})')

# Plot the observed chi-squared statistic
plt.axvline(
    x=mcnemar_statistic,
    color='limegreen', linestyle='-.',
    label=f"M² ({mcnemar_statistic:.3f})")

# Shade the P-value area
plt.fill_between(
    x[x >= mcnemar_statistic],
    hx[x >= mcnemar_statistic],
    color='greenyellow',
    label=f'P ({p_value_mcnemar:.4f})')

# Add labels and title
plt.xlabel('χ²')
plt.ylabel('Density')
plt.ylim((0, .5))
plt.margins(x=0.02, y=0)
plt.yticks([])
plt.title(f"χ² distribution (DF=1)")
plt.legend();

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

print("Results of the McNemar test (statsmodels):")
print(
    mcnemar(
        table_mcnemar,  # exact=True uses the binomial distribution
        exact=False,    # exact=False uses the χ² distribution
        correction=True))

#### Bayesian approach


In [ ]:
# Perform Bayesian estimation of the mean difference
bayes_result = stats.bayes_mvs(differences_fertil, alpha=0.95)

# Extract and print the results
posterior_mean = bayes_result[0][0]
credible_interval = np.array(bayes_result[0][1])

print(f"Bayesian estimate of the paired difference mean = \
{posterior_mean:.3f}")
print(f"Associated 95% credible interval:", credible_interval.round(3))

## Non-parametric approach 


### Wilcoxon signed-rank test
#### Exact method
#### Ties and zeros


In [ ]:
from tabulate import tabulate
from scipy.stats import rankdata

# Extract the signs of the differences
# encoding 1 for positive, -1 for negative, 0 for zero
signs = np.sign(differences_fertil)

# Calculate ranks of absolute differences
ranks_abs_differences = rankdata(np.absolute(differences_fertil))

# Calculate the sum of ranks for positive and negative differences
W_pos = np.sum(ranks_abs_differences[differences_fertil > 0])
W_neg = np.sum(ranks_abs_differences[differences_fertil < 0])

# Compute the W-statistic as the minimum of W_pos and W_neg
W_statistic = min(W_pos, W_neg)  # Corrected calculation

# Critical W value (two-tailed) for alpha=0.05 and n=15 from table
W_critical = 25

# Print the detailed results
print("Wilcoxon signed-rank test - Manual calculation")
print()
print("1. Difference, sign, and rank for each matched pair:")
# Pretty output (but not mandatory)
print(
    tabulate(
        {
            "Differences": differences_fertil,
            "Signs": signs,
            "Ranks": ranks_abs_differences,
        },
        headers='keys', tablefmt='pretty',
    )
)
print()
print("2. Rank Sums:")
print(f" Sum of ranks for positive differences (W+) = {W_pos}")
print(f" Sum of ranks for negative differences (W-) = {W_neg}")
print()
print("3. W-Statistic calculation:")
print(f" W-statistic (min(W+, W-)) = {W_statistic}")
print()
print("4. Comparison with critical value:")
print(f" Critical W value (n={len(differences_fertil)}, α=0.05, two-tailed) \
= {W_critical}")
print()

# Compare the W-statistic to the critical value and draw a conclusion
if W_statistic <= W_critical:
    print("Conclusion: reject the null hypothesis.")
    print(" There is a statistically significant difference \
between the paired groups.")
else:
    print("Conclusion: fail to reject the null hypothesis.")
    print(" There is no statistically significant difference between \
the paired groups.")

#### Asymptotic approach


In [ ]:

from scipy.stats import norm

n = differences_fertil[differences_fertil != 0].size  # Non-zero differences

# Significance level (alpha)
α = 0.05

# Determine the mean and standard deviation for the normal approximation
μ = n * (n + 1) / 4
σ = np.sqrt(n * (n + 1) * (2 * n + 1) / 24)

# Generate x values for plotting the normal distribution
x = np.linspace(μ - 4 * σ , μ + 4 * σ , 1000)
normal_pdf = norm.pdf(x, loc=μ, scale=σ)

# Create the plot
plt.figure(figsize=(3.5, 3))
plt.plot(x, normal_pdf, lw=2, color="black")

# Plot the critical W-values (two-tailed test)
W_crit_lower = μ - norm.ppf(1 - α/2) * σ   # Lower critical W
W_crit_upper = μ + norm.ppf(1 - α/2) * σ   # Upper critical W

plt.axvline(
    x=W_crit_lower,
    color='orangered', linestyle='--')
plt.axvline(
    x=W_crit_upper,
    color='orangered', linestyle='--',
    label=f"W* ({W_crit_lower:.3f})")

# Shade the rejection regions (alpha)
plt.fill_between(
    x, normal_pdf,
    where=(x <= W_crit_lower) | (x >= W_crit_upper),
    linestyle="-", linewidth=2, color='tomato',
    label=f"α ({α})")

# Calculate the z-score
W_crit = (W_statistic - μ) / σ

# Calculate the P value (two-tailed)
p_value_W = 2 * (1 - norm.cdf(abs(W_crit)))

# Plot the observed W-statistic
plt.axvline(
    x=float(W_statistic),
    color='limegreen', linestyle='-.',
    label=f"W ({W_statistic:n})")

# Shade the P value areas (two-tailed)
plt.fill_between(
    x, normal_pdf,
    where=(x <= W_statistic) | (x >= μ + (μ - W_statistic)),
    color='greenyellow',
    label=f"P ({p_value_W:.5f})")

# Add labels and title
plt.xlabel('W')
plt.ylabel('Probability density')
plt.margins(x=0, y=0)
plt.yticks([])
plt.title(f"Normal approximation to W")
plt.legend();

### Python tools for the Wilcoxon signed-rank test


In [ ]:
from scipy.stats import wilcoxon

# Perform Wilcoxon signed-rank test with exact P value calculation
W_statistic_exact_scipy, p_value_exact_scipy = wilcoxon(
    differences_fertil, method='exact')

# Perform Wilcoxon signed-rank test with normal approximation
W_statistic_asymptotic_scipy, p_value_asymptotic_scipy = wilcoxon(
    differences_fertil, method='approx', correction=False)

# Perform Wilcoxon signed-rank test using Pingouin
wilcoxon_pingouin = pg.wilcoxon(
    differences_fertil,
    method='approx',
    correction=False,  # Pingouin applies a continuity correction by default
)

# Print all the results
print("Wilcoxon signed-rank test results (SciPy):")
print(f" Exact P value = {p_value_exact_scipy:.5f}")
print(f" Asymptotic P value (normal approximation) = \
{p_value_asymptotic_scipy:.5f}")
print()
print("Wilcoxon signed-rank test results (Pingouin):")
print(wilcoxon_pingouin.round(5))

## Bootstrapping and permutation tests for paired data
### Generating bootstrap pair samples


In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Generate 10,000 bootstrap replicates of paired mean differences
n_replicates = 10000
bs_paired_mean_diff = np.array([
    np.mean(
        np.random.choice(
            differences_fertil,
            size=len(differences_fertil),
            replace=True
        )) for _ in range(n_replicates)
    ])
# We cold have used for loop instead of list comprehension

print("First ten calculated bootstrap paired mean differences:")
print(bs_paired_mean_diff[:10].round(3))

### Estimate of the confidence interval for the paired mean differences
#### Exact boostrap percentile interval


In [ ]:
# Calculate the 95% percentile interval
bs_m = bs_paired_mean_diff.mean()
bs_ci = np.percentile(bs_paired_mean_diff, [2.5, 97.5])

# Print the results
print(
    f"Bootstrap estimate of the mean of paired differences = {bs_m:5.2f}")
print(f"95% bootstrap percentile interval = {bs_ci.round(3)}")

In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the histogram of the bootstrap distribution of mean paired differences
plt.hist(
    bs_paired_mean_diff,
    density=False, bins='auto',
    color='gold',
    label='Bootstrap replicates')

# Annotate the observed mean difference
plt.axvline(
    x=differences_fertil_mean, 
    color='cornflowerblue', linestyle='-', lw=2,
    label=f'Observed estimate ({differences_fertil_mean:.3f})')

# Add lines for the confidence interval
plt.axvline(
    x=bs_ci[0],
    color='red', linestyle='--',
    label=f'2.5th ({bs_ci[0]:.2f})')
plt.axvline(
    x=bs_ci[1],
    color='red', linestyle='--',
    label=f'97.5th ({bs_ci[1]:.2f})')

# Add labels and title
plt.xlabel('Paired mean difference')
plt.ylabel('Frequency')
plt.title(f"Bootstrap paired mean difference")
plt.legend();

#### Asymptotic confidence interval


In [ ]:
# Calculate the 95% normal confidence interval
bs_s = bs_paired_mean_diff.std(ddof=1)
z_crit = norm.ppf(1 - α/2)

bs_ci_normal = np.array((
    bs_m - z_crit*bs_s,
    bs_m + z_crit*bs_s
))

# Print the results
print(f"Bootstrap standard error = {bs_s:.5f}")
print(f"95% asymptotic CI of bootstrap distribution = \
{bs_ci_normal.round(3)}")

### Bootstrapping with Pingouin


In [ ]:
# Calculate 95% percentile bootstrap confidence interval using Pingouin
bs_ci_pg, bt_pg = pg.compute_bootci(
    x=differences_fertil,  # Univariate array of paired differences
    func='mean',
    method='per',
    seed=1234,
    n_boot=10000,
    return_dist=True)

print(
    "First eight calculated bootstrap paired mean differences (Pingouin):")
print(bt_pg[:8].round(2))
print()
print(
    "95% percentile bootstrap percentile interval (Pingouin):",
    bs_ci_pg.round(3))

### Bootstrap P value for paired data
#### Shifted t-statistics


In [ ]:
# Set the random seed for reproducibility
np.random.seed(111)

n_replicates = 10**4 - 1  # 99 rule

# Shift the paired differences to have a mean of zero (simulating H0)
shifted_differences = differences_fertil - differences_fertil.mean()

# Generate B bootstrap replicates of the shifted differences
bs_shifted_differences = np.array([
    np.random.choice(
        shifted_differences, size=differences_fertil.size, replace=True)
    for _ in range(n_replicates)])

# Calculate the mean for each shifted replicate
bs_shifted_differences_mean = bs_shifted_differences.mean(axis=1)
# Calculate the sample standard deviation (ddof=1)
bs_shifted_differences_std = bs_shifted_differences.std(ddof=1, axis=1)
# Calculate the standard error for each replicate
bs_se = bs_shifted_differences_std / differences_fertil.size**.5

# Calculate the t-statistic for each shifted replicate
bs_t_statistic = bs_shifted_differences_mean / bs_se

# Or simply calculate the t-test for the mean 
# on the one-sample bootstrap distribution
# bs_t_statistic = np.array([
#     stats.ttest_1samp(x, popmean=0)[0] for x in bs_shifted_differences
# ])

print("First ten calculated shifted paired differences t-statistics:")
print(bs_t_statistic[:10].round(2))

#### One-tailed P value


In [ ]:
# Calculate one-sided P value using mean shifting, considering the 
# direction of the observed t-statistic.
# # 1. Calculate lower and upper tail
if t_statistic_paired >= 0:
    # p_value_bs_shifted_low = np.sum(bs_t_statistic >= t_statistic_paired) 
    # / len(bs_t_statistic)
    p_value_bs_shifted_up = np.mean(bs_t_statistic >= t_statistic_paired)
    p_value_bs_shifted_lo = np.mean(bs_t_statistic <= -t_statistic_paired)
else:
    p_value_bs_shifted_lo = np.mean(bs_t_statistic <= t_statistic_paired)
    p_value_bs_shifted_up = np.mean(bs_t_statistic >= -t_statistic_paired)

# 2. Attribute the one-tailed P value
p_value_bs_shifted_1t = p_value_bs_shifted_up if t_statistic_paired >= 0 \
else p_value_bs_shifted_lo

# Print the P value
print(f"One-sided P value obtained using shifted paired-sample t-statistics\
 = {p_value_bs_shifted_1t:.4f}")

In [ ]:

import matplotlib.patches as mpatches

plt.figure(figsize=(3.5, 3))

# Plot the histogram of the bootstrap t-statistics
hist, bins, patches = plt.hist(
    bs_t_statistic,
    density=False,
    bins=int(n_replicates**.5),
    color='gold',)

# Annotate the observed t-statistic
plt.axvline(
    x=t_statistic_paired,
    color='limegreen', linestyle='-.', lw=2,
    label=f'Observed t ({t_statistic_paired:.3f})')

# Determine the direction of the observed t-statistic and plot accordingly
if t_statistic_paired >= 0:
    # histogram of the bootstrap t-statistics >= observed t-statistic
    extreme_t_stats = bs_t_statistic[bs_t_statistic >= t_statistic_paired]
    # Change the color of the bars based on the direction parameter
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge >= extreme_t_stats):
            patches[i].set_facecolor('lime')
else:
    # histogram of the bootstrap t-statistics <= observed t-statistic
    extreme_t_stats = bs_t_statistic[bs_t_statistic <= t_statistic_paired]
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge <= extreme_t_stats):
            patches[i].set_facecolor('lime')

# Add labels and title
plt.xlabel('t')
plt.ylabel('Frequency')
plt.title(f"Shifted t-statistic under H0")

# Create a copy of the original patch for the legend
original_patch = mpatches.Patch(color='gold', label='Bootstrap t')
# Create a patch for the legend
p_value_patch = mpatches.Patch(
    color='lime', label=f'One-tailed P ({p_value_bs_shifted_1t:.3f})')

# Add the patches to the legend
plt.legend(handles=[original_patch, plt.gca().lines[0], p_value_patch]);

#### Estimating two-tailed shifted P value


In [ ]:
# Maximum one-tail
p_value_bs_shifted_2t_max = 2 * max(
    p_value_bs_shifted_lo, p_value_bs_shifted_up)

# Sum of the tails
p_value_bs_shifted_2t_sum = p_value_bs_shifted_lo + p_value_bs_shifted_up

# Print the results
print("Two-tailed paired-sample t-test bootstrap (shifted) P value")
print(f"(conservative) = {p_value_bs_shifted_2t_max:.4f}")
print(f"Two-tailed paired-sample t-test bootstrap (shifted) P value")
print(f"(both tails) = {p_value_bs_shifted_2t_sum:.4f}")

#### Permutation


In [ ]:
def permute_paired_differences(differences):
    """
    Generates a permuted sample of paired differences under H0
    
    Args:
        differences: an array of paired differences.

    Returns:
        A new array with randomly negated differences within each pair.
    """

    # Randomly generate -1 or 1 for each pair, i.e., for each pair
    # either we reverse the difference or not
    signs = np.random.choice([-1, 1], size=len(differences))

    # Multiply each difference by its corresponding sign
    permuted_differences = differences * signs
    return permuted_differences

In [ ]:
# Set random seed for reproducibility
np.random.seed(111)

# Generate bootstrap replicates of the mean difference 
# for each permutation sample
n_replicates = 9999

# Initialize an empty array to store the permuted mean differences
permuted_differences_mean = np.zeros(n_replicates)

# Generate permuted mean differences
for i in range(n_replicates):
    # Permute the differences
    permuted_differences = permute_paired_differences(differences_fertil)

    # Calculate the mean difference for this permuted sample
    permuted_differences_mean[i] = np.mean(permuted_differences)

# We could also use a list comprehension instead of the loop
# permuted_differences_mean = np.array([
#     np.mean(
#         permute_paired_differences(
#             differences_fertil)) for _ in range(n_replicates)])

print("Firt ten mean differences of the permutation replicates:")
print(permuted_differences_mean[:10].round(2))

In [ ]:
# Calculate one-sided P value using permutation distribution 
# of paired mean differences
if differences_fertil_mean >= 0:
    p_value_permutation_up = np.mean(
        permuted_differences_mean >= differences_fertil_mean)
    p_value_permutation_lo = np.mean(
        permuted_differences_mean <= -differences_fertil_mean)
else:
    p_value_permutation_lo = np.mean(
        permuted_differences_mean <= differences_fertil_mean)
    p_value_permutation_up = np.mean(
        permuted_differences_mean >= -differences_fertil_mean)

p_value_permutation_1t = (
    p_value_permutation_up if differences_fertil_mean >= 0
    else p_value_permutation_lo)

# Print the P value
print(f"One-sided P value obtained using permutation test \
(paired mean difference) = {p_value_permutation_1t:.4f}")

In [ ]:

plt.figure(figsize=(3.5, 3))

# Plot the histogram of the permuted mean differences
hist, bins, patches = plt.hist(
    permuted_differences_mean,
    density=False, bins=int(n_replicates**.5),
    color='gold')

# Annotate the observed mean difference
plt.axvline(
    x=differences_fertil_mean, 
    color='limegreen', linestyle='-.', lw=2,
    label=f'Observed estimate ({differences_fertil_mean:.3f})')

# Determine the direction of observed mean difference and plot accordingly
if differences_fertil_mean >= 0:
    # Plot the histogram of the mean differences >= observed mean difference 
    extreme_diffs = permuted_differences_mean[
        permuted_differences_mean >= differences_fertil_mean]
    # Change the color of the bars based on the direction parameter
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge >= extreme_diffs):
            patches[i].set_facecolor('lime')
else:
    # Plot the histogram of the mean differences <= observed mean difference
    extreme_diffs = permuted_differences_mean[
        permuted_differences_mean <= differences_fertil_mean]
    for i, bin_edge in enumerate(bins[:-1]):
        if np.any(bin_edge <= extreme_diffs):
            patches[i].set_facecolor('lime')

# Add labels and title
plt.xlabel('Paired mean difference')
plt.ylabel('Frequency')
plt.title(f"Permutation distribution under H0")

# Create a copy of the original patch for the legend
original_patch = mpatches.Patch(color='gold', label='Permuted replicates')
# Create a patch for the legend
p_value_patch = mpatches.Patch(
    color='lime', label=f'One-tailed P ({p_value_permutation_1t:.3f})')

# Add the patches to the legend
plt.legend(handles=[original_patch, plt.gca().lines[0], p_value_patch]);

In [ ]:
# Maximum one-tail
p_value_permut_2t_max = 2 * max(
    p_value_permutation_lo, p_value_permutation_up)

# Sum of the tails
p_value_permut_2t_sum = p_value_permutation_lo + p_value_permutation_up

# Print the results
print(f"Two-tailed permutation P value (conservative) = \
{p_value_permut_2t_max:.4f}")
print(f"Two-tailed permutation P value (both tails) = \
{p_value_permut_2t_sum:.4f}")

## Conclusion
